# Residual Networks and Normalization


Figures are from Prince, *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*,
unless another source is cited. Code cells adapted from the UDL notebooks (MIT
license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).


## Topics

- Residual connections and residual blocks
- Exploding gradients in residual networks
- Batch normalization
- Common residual architectures
- Normalization in transformers: LayerNorm, RMSNorm, pre-norm vs. post-norm, QK-norm

# Residual connections and residual blocks

We start from the sequential networks of earlier lectures, see why simply making
them deeper stops helping, and introduce residual connections as the fix.


## Previously we saw a sequential network:

$$
\begin{aligned}
\mathbf{h}_1 &= \mathbf{f}_1[\mathbf{x}, \boldsymbol{\phi}_1] \\
\mathbf{h}_2 &= \mathbf{f}_2[\mathbf{h}_1, \boldsymbol{\phi}_2] \\
\mathbf{h}_3 &= \mathbf{f}_3[\mathbf{h}_2, \boldsymbol{\phi}_3] \\
\mathbf{y} &= \mathbf{f}_4[\mathbf{h}_3, \boldsymbol{\phi}_4]
\end{aligned}
$$


![](figs/08-Residual-Networks-and-Normalization/sequential-network.png)


Fully connected network:

$$
h_i = \mathrm{a}\left[\beta_i + \sum_{j=1}^{D} \omega_{ij} x_j\right]
$$


Convolutional network (e.g. 1 channel $\rightarrow$ 1 channel):

$$
\begin{aligned}
h_i &= \mathrm{a}\left[\beta + \omega_1 x_{i-1} + \omega_2 x_i + \omega_3 x_{i+1}\right] \\
&= \mathrm{a}\left[\beta + \sum_{j=1}^{3} \omega_j x_{i+j-2}\right]
\end{aligned}
$$


## Previously we saw a sequential network (cont.)

![](figs/08-Residual-Networks-and-Normalization/sequential-network.png)

Can think of as a sequence of nested functions:

$$
\mathbf{y} = \mathbf{f}_4\Big[\mathbf{f}_3\Big[\mathbf{f}_2\big[\mathbf{f}_1[\mathbf{x}, \boldsymbol{\phi}_1], \boldsymbol{\phi}_2\big], \boldsymbol{\phi}_3\Big], \boldsymbol{\phi}_4\Big]
$$

## More layers are better…

![ImageNet top-1 error by year. AlexNet has 8 layers; VGG-19 has 19 weight layers (the figure's label says 18).](figs/08-Residual-Networks-and-Normalization/imagenet-more-layers.png)

## More layers are better… up to a point

![Convolutional network on CIFAR10.](figs/08-Residual-Networks-and-Normalization/cifar10-depth-test-train-error.png)

## What's going on?

Not completely understood, but…

Take a look at $\partial y / \partial x$ for shallow and deep networks.

![](figs/08-Residual-Networks-and-Normalization/shattered-gradients.png)

- Gradients of deeper networks are much less correlated! (c: autocorrelation)
- A small step in gradient descent may jump to wildly different valued gradient!

***The Shattered Gradient Phenomenon***
[(Balduzzi et al., 2017, arXiv:1702.08591)](https://arxiv.org/abs/1702.08591)


## What's going on? *The Shattered Gradient Phenomenon*

![](figs/08-Residual-Networks-and-Normalization/sequential-network-unlabeled.png)

$$
\mathbf{y} = \mathbf{f}_4\Big[\mathbf{f}_3\Big[\mathbf{f}_2\big[\mathbf{f}_1[\mathbf{x}, \boldsymbol{\phi}_1], \boldsymbol{\phi}_2\big], \boldsymbol{\phi}_3\Big], \boldsymbol{\phi}_4\Big]
$$

The derivative of the output $y$ w.r.t. the first layer $\mathrm{f}_1$ is, by the chain rule:

$$
\frac{\partial \mathbf{y}}{\partial \mathbf{f}_1} = \frac{\partial \mathbf{f}_4}{\partial \mathbf{f}_3} \frac{\partial \mathbf{f}_3}{\partial \mathbf{f}_2} \frac{\partial \mathbf{f}_2}{\partial \mathbf{f}_1}
$$

$\mathrm{f}_1$ impacts $\mathrm{f}_2$ impacts $\mathrm{f}_3$, etc…

## Solution: Residual connections

**Regular sequential network:**

$$
\begin{aligned}
\mathbf{h}_1 &= \mathbf{f}_1[\mathbf{x}, \boldsymbol{\phi}_1] \\
\mathbf{h}_2 &= \mathbf{f}_2[\mathbf{h}_1, \boldsymbol{\phi}_2] \\
\mathbf{h}_3 &= \mathbf{f}_3[\mathbf{h}_2, \boldsymbol{\phi}_3] \\
\mathbf{y} &= \mathbf{f}_4[\mathbf{h}_3, \boldsymbol{\phi}_4]
\end{aligned}
$$


![](figs/08-Residual-Networks-and-Normalization/sequential-network.png)


**Residual network:**

$$
\begin{aligned}
\mathbf{h}_1 &= \mathbf{x} + \mathbf{f}_1[\mathbf{x}, \boldsymbol{\phi}_1] \\
\mathbf{h}_2 &= \mathbf{h}_1 + \mathbf{f}_2[\mathbf{h}_1, \boldsymbol{\phi}_2] \\
\mathbf{h}_3 &= \mathbf{h}_2 + \mathbf{f}_3[\mathbf{h}_2, \boldsymbol{\phi}_3] \\
\mathbf{y} &= \mathbf{h}_3 + \mathbf{f}_4[\mathbf{h}_3, \boldsymbol{\phi}_4]
\end{aligned}
$$


![](figs/08-Residual-Networks-and-Normalization/residual-connections.png)


*[K. He, X. Zhang, S. Ren, and J. Sun, "Deep Residual Learning for Image Recognition," arXiv:1512.03385, Dec. 2015](http://arxiv.org/abs/1512.03385)*

## Residual Network


Substituting in:

$$
\begin{aligned}
\mathbf{y} = \mathbf{x} \;+\;& \mathbf{f}_1[\mathbf{x}] \\
+\;& \mathbf{f}_2\big[\mathbf{x} + \mathbf{f}_1[\mathbf{x}]\big] \\
+\;& \mathbf{f}_3\Big[\mathbf{x} + \mathbf{f}_1[\mathbf{x}] + \mathbf{f}_2\big[\mathbf{x} + \mathbf{f}_1[\mathbf{x}]\big]\Big] \\
+\;& \mathbf{f}_4\Big[\mathbf{x} + \mathbf{f}_1[\mathbf{x}] + \mathbf{f}_2\big[\mathbf{x} + \mathbf{f}_1[\mathbf{x}]\big] + \mathbf{f}_3\Big[\mathbf{x} + \mathbf{f}_1[\mathbf{x}] + \mathbf{f}_2\big[\mathbf{x} + \mathbf{f}_1[\mathbf{x}]\big]\Big]\Big]
\end{aligned}
$$

## Residual Network (cont.)

We can unravel all the possible paths

The output is the sum of the input plus 4 partial networks.


![](figs/08-Residual-Networks-and-Normalization/residual-unraveled.png)


## Residual Network as Ensemble of Networks

![](figs/08-Residual-Networks-and-Normalization/residual-unraveled.png)


- Ensemble of four networks (the paths summed at the output $\mathbf{y}$)
- Another ensemble of four networks (the teal paths summed before $\mathbf{f}_4$)

- 16 possible paths through the network!
- 8 paths include $\mathrm{f}_1$
- The influence of $\mathrm{f}_1$ on $\partial y / \partial f_1$ takes 8 different forms
- Gradients on shorter paths generally better behaved.


- During training, the model can amplify or attenuate the different paths to achieve the best results


$$
\frac{\partial \mathbf{y}}{\partial \mathbf{f}_1} = \mathbf{I} + \frac{\partial \mathbf{f}_2}{\partial \mathbf{f}_1} + \left(\frac{\partial \mathbf{f}_3}{\partial \mathbf{f}_1} + \frac{\partial \mathbf{f}_3}{\partial \mathbf{f}_2}\frac{\partial \mathbf{f}_2}{\partial \mathbf{f}_1}\right) + \left(\frac{\partial \mathbf{f}_4}{\partial \mathbf{f}_1} + \frac{\partial \mathbf{f}_4}{\partial \mathbf{f}_2}\frac{\partial \mathbf{f}_2}{\partial \mathbf{f}_1} + \frac{\partial \mathbf{f}_4}{\partial \mathbf{f}_3}\frac{\partial \mathbf{f}_3}{\partial \mathbf{f}_1} + \frac{\partial \mathbf{f}_4}{\partial \mathbf{f}_3}\frac{\partial \mathbf{f}_3}{\partial \mathbf{f}_2}\frac{\partial \mathbf{f}_2}{\partial \mathbf{f}_1}\right)
$$


Each path through $K$ residual blocks either enters or skips each block, so
there are $2^K$ paths, and the number that pass through exactly $n$ blocks is
the binomial coefficient $\binom{K}{n}$. Half of them pass through any given
block, which is why 8 of the 16 paths include $\mathbf{f}_1$ (and the
derivative above has 8 terms). Change `K`: the right panel uses $K = 54$, the
number of residual blocks in a 110-layer CIFAR ResNet, where almost every path
is about half as deep as the network.

In [ ]:
#| code-fold: true
#| fig-cap: "Fraction of the paths through a residual network that pass through $n$ of its $K$ blocks."
#| fig-alt: "Two bar charts of the fraction of paths against the number of blocks on the path. Left, K = 4: bars 1/16, 4/16, 6/16, 4/16, 1/16 for 0 to 4 blocks. Right, K = 54: a narrow bell shape centered at 27 blocks, with essentially no paths shorter than 15 or longer than 40 blocks."
from math import comb
import numpy as np
import matplotlib.pyplot as plt

K = 4   # number of residual blocks

print(f"K = {K} residual blocks")
print(f"paths from x to y:            {2 ** K}")
print(f"paths that pass through f1:   {2 ** (K - 1)}")
for n in range(K + 1):
    print(f"  paths through {n} of the {K} blocks: {comb(K, n)}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, k in zip(axes, [K, 54]):
    lengths = np.arange(k + 1)
    fraction = np.array([comb(k, n) for n in lengths]) / 2 ** k
    ax.bar(lengths, fraction, color="#2a78d6", width=0.8)
    ax.set_xlabel("blocks on the path")
    ax.set_ylabel("fraction of paths")
    ax.set_title(f"K = {k}: {2 ** k:,} paths" if k < 20
                 else f"K = {k}: $2^{{{k}}} \\approx$ {2 ** k:.1e} paths")
fig.tight_layout()
plt.show()

## Order of operations is important

![](figs/08-Residual-Networks-and-Normalization/residual-block-orderings.png)


**a)** Can only add to the residual because of the ReLU

**b)** More flexible approach to end with linear block. Starting with linear block gives us some flexibility on spatial resolution.

**c)** Note: if we start with a ReLU, then will clamp negative values and so do nothing


# Exploding gradients in residual networks

Residual connections let us train much deeper networks, but this helps only up
to a point: they bring a new problem of their own, which the rest of the
lecture is about.


## Exploding Gradients in Residual Networks


![](figs/08-Residual-Networks-and-Normalization/residual-variance-renormalized.png)


- With He initialization we can control the variance inside the block
- But variance doubles when we add the residual back in.
- And then grows exponentially.

- Could stabilize by renormalizing after adding each residual.
- More common to apply *batch normalization*.


# Batch normalization

Batch normalization is the usual way to keep the variance under control in a
residual *convolutional* network. Later in the lecture we will see that
transformers normalize along a different axis, but the recipe (standardize,
then rescale and shift with learned parameters) is the same.


## Batch Normalization (a.k.a. *BatchNorm*)

![](figs/08-Residual-Networks-and-Normalization/batchnorm-residual-network.png)

- Shifts and rescales each activation so that its mean and variance across the batch become values that are learned during training

Calculate the sample *mean* and *standard deviation* for each hidden unit across samples of the batch.

$$
\begin{aligned}
m_h &= \frac{1}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} h_i \\
s_h &= \sqrt{\frac{1}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} (h_i - m_h)^2}
\end{aligned}
$$


*Standardize* (*normalize*) to zero-mean and unit standard deviation.

$$
\hat{h}_i \leftarrow \frac{h_i - m_h}{s_h + \epsilon} \qquad \forall i \in \mathcal{B},
$$


Scale by $\gamma$ and shift by $\delta$, which are *learned* parameters.

$$
h_i \leftarrow \gamma \hat{h}_i + \delta \qquad \forall i \in \mathcal{B}.
$$


*[S. Ioffe and C. Szegedy, "Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift," arXiv:1502.03167, Mar. 2015](http://arxiv.org/abs/1502.03167)*


## Batch Normalization (cont.)


- Applied independently to each hidden unit

- **Standard FC Network** with $K$ layers, each with $D$ hidden units:
  - $KD$ learned scales, $\gamma$, and $KD$ learned offset, $\delta$

- **Convolutional Network** with $K$ layers, each with $C$ channels:
  - $KC$ learned scales, $\gamma$, and $KC$ learned offset, $\delta$
  - one $m_h, s_h$ per channel, averaged over the batch *and* all spatial positions

## BatchNorm at training time vs. inference time

**Training:** $m_h, s_h$ are computed from the current mini-batch, so every
example's output depends on which other examples share its batch.

- This injects noise: the scale and shift jitter from batch to batch
- $\rightarrow$ acts as a **regularizer** (one reason BatchNorm networks need less dropout)
- Side effect: results depend on batch size; very small batches give noisy $m_h, s_h$


**Inference:** there may be no batch (one image at a time), and the output of
an example should not depend on its neighbors.

- Keep **running averages** of $m_h$ and $s_h^2$ during training (exponential moving average, PyTorch default momentum 0.1)
- At inference use the stored values: BatchNorm becomes a fixed per-channel affine map $h \leftarrow \gamma \frac{h - m_h}{s_h + \epsilon} + \delta$
- ... which can be *folded into the preceding convolution*, so BatchNorm costs nothing at inference


Remember to call `model.eval()`: in `train()` mode PyTorch keeps using batch statistics and keeps updating the running averages.


The running averages for one hidden unit, updated as PyTorch does:
$\bar{m} \leftarrow 0.9\,\bar{m} + 0.1\,m_h$ and the same for $s_h^2$,
starting from $\bar{m} = 0$, $\bar{s}^2 = 1$. The weight left on the
starting value after $t$ batches is $0.9^t$, so after a few dozen batches the
averages track the true statistics, still jittering by about a quarter of the
batch-to-batch noise ($\sqrt{0.1/1.9} \approx 0.23$). The cell then folds the inference-time BatchNorm into
the linear layer before it and checks that the outputs agree.

In [ ]:
#| code-fold: true
#| fig-cap: "Batch statistics (gray) and their running averages with momentum 0.1 (blue) for one hidden unit whose true mean is 3 and variance 4."
#| fig-alt: "Two panels against training step 1 to 60. Left: batch means scattered between about 2 and 4, and a running mean that rises from near 0 to about 3 by step 30 and then stays near the dashed true value 3. Right: batch variances scattered between about 2 and 6, and a running variance that rises from near 1 to about 4 by step 30 and stays near the dashed true value 4."
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
true_mean, true_std = 3.0, 2.0   # the hidden unit's distribution during training
B = 32                           # batch size
momentum = 0.1                   # PyTorch default
steps = 60
eps = 1e-5

running_mean, running_var = 0.0, 1.0      # PyTorch's initial values
history = []
for t in range(steps):
    h = rng.normal(true_mean, true_std, size=B)
    m_h, s2_h = h.mean(), h.var(ddof=1)   # PyTorch stores the unbiased variance
    running_mean = (1 - momentum) * running_mean + momentum * m_h
    running_var = (1 - momentum) * running_var + momentum * s2_h
    history.append((m_h, s2_h, running_mean, running_var))
history = np.array(history)
print(f"after {steps} batches: running mean = {running_mean:.3f} (true {true_mean}), "
      f"running var = {running_var:.3f} (true {true_std**2})")
print(f"weight left on the initial values: 0.9^{steps} = {0.9**steps:.4f}")

# At inference BatchNorm is a fixed affine map, so it folds into the preceding layer
gamma, delta = 1.5, -0.5                  # learned scale and shift
w, b = 0.8, 0.2                           # preceding layer: h = w * x + b
s_h = np.sqrt(running_var)
w_folded = gamma * w / (s_h + eps)
b_folded = gamma * (b - running_mean) / (s_h + eps) + delta
x = 4.0                                   # one example, no batch
bn_out = gamma * ((w * x + b) - running_mean) / (s_h + eps) + delta
print(f"linear then BatchNorm: {bn_out:.6f}   folded linear layer: {w_folded * x + b_folded:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
t = np.arange(1, steps + 1)
for ax, (i_batch, i_run), true, label in [
        (axes[0], (0, 2), true_mean, "$m_h$"), (axes[1], (1, 3), true_std**2, "$s_h^2$")]:
    ax.plot(t, history[:, i_batch], ".", color="#8a8984", label=f"batch {label}")
    ax.plot(t, history[:, i_run], "-", color="#2a78d6", label=f"running {label}")
    ax.axhline(true, ls="--", color="#eb6834", label="true value")
    ax.set_xlabel("training step")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Benefits of BatchNorm

![](figs/08-Residual-Networks-and-Normalization/batchnorm-variance-growth.png)

**Stable forward propagation**

- Initialize offsets $\delta$ to zero and scales $\gamma$ to 1
- Variance now increases linearly
- $k^{th}$ block adds *one unit of variance* to variance of $k$
- At initialization, later layers make smaller relative change to overall variation
- During training, the scales can increase in later layers if helpful
  - $\rightarrow$ control the effective depth

The variance argument of the last few slides, checked numerically: 16
residual blocks at He initialization, each branch a ReLU followed by a linear
layer (ordering b), pushed forward on a batch of 512 random inputs. The cell
measures the mean square $\mathbb{E}[h^2]$ of the residual stream, the
quantity He initialization preserves (the stream has zero mean over random
weights, so this is its variance). Without normalization it doubles at every
block; dividing by $\sqrt{2}$ after each addition keeps it at 1; BatchNorm at
the start of each branch makes every branch contribute one unit, so it grows
as $k + 1$.

In [ ]:
#| code-fold: true
#| fig-cap: "Variance of the residual stream after each of 16 residual blocks at initialization, on a log scale, for three ways of combining the branch with the stream."
#| fig-alt: "Log-scale plot of variance against residual block k from 0 to 16. The plain residual network (orange) follows the dotted line 2 to the k, reaching about 2 to the 15.6. The renormalized network (green) stays near 1. The network with BatchNorm in each branch (blue) follows the dashed line k + 1, reaching about 17."
# Adapted from UDL notebook 11.3 (MIT)
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
K = 16       # residual blocks
D = 256      # hidden units per layer
B = 512      # batch size
eps = 1e-5

def he_linear(d_in, d_out):
    return rng.normal(0, np.sqrt(2 / d_in), size=(d_in, d_out))

def batchnorm(h, gamma=1.0, delta=0.0):     # per hidden unit, over the batch
    m_h, s_h = h.mean(axis=0), h.std(axis=0)
    return gamma * (h - m_h) / (s_h + eps) + delta

def branch(h, W):                           # ReLU then linear
    return np.maximum(h, 0) @ W

variants = {
    "plain": lambda h, W: h + branch(h, W),
    "renormalized": lambda h, W: (h + branch(h, W)) / np.sqrt(2),
    "BatchNorm": lambda h, W: h + branch(batchnorm(h), W),
}
labels = {"plain": r"$\mathbf{h} + \mathbf{f}[\mathbf{h}]$",
          "renormalized": r"$(\mathbf{h} + \mathbf{f}[\mathbf{h}])/\sqrt{2}$",
          "BatchNorm": r"$\mathbf{h} + \mathbf{f}[\mathrm{BN}[\mathbf{h}]]$"}

weights = [he_linear(D, D) for _ in range(K)]
h0 = rng.normal(size=(B, D))                # variance 1 entering the first block

variance = {}
for name, block in variants.items():
    h, v = h0, [np.mean(h0 ** 2)]
    for W in weights:
        h = block(h, W)
        v.append(np.mean(h ** 2))
    variance[name] = np.array(v)

print("variance of the residual stream after k blocks")
print("k   " + "".join(f"{name:>14}" for name in variants))
for k in [0, 1, 2, 3, 4, 8, 16]:
    print(f"{k:<4}" + "".join(f"{variance[name][k]:14.1f}" for name in variants))

k = np.arange(K + 1)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for (name, v), c in zip(variance.items(), ["#eb6834", "#1baf7a", "#2a78d6"]):
    ax.plot(k, v, "o-", color=c, ms=4, label=labels[name])
ax.plot(k, 2.0 ** k, ":", color="gray", label="$2^k$")
ax.plot(k, k + 1.0, "--", color="gray", label="$k+1$")
ax.set_yscale("log", base=2)
ax.set_xlabel("residual block $k$")
ax.set_ylabel("variance of the residual stream")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Benefits of BatchNorm (cont.)

**Supports higher learning rates**

- Ioffe & Szegedy attributed this to reducing "internal covariate shift"
- [Santurkar et al. (2018, arXiv:1805.11604)](https://arxiv.org/abs/1805.11604)
  showed the actual mechanism is that BatchNorm makes the loss surface
  *smoother*: the loss and its gradient change more slowly along the
  gradient direction, so a larger step is safe

**Regularization via added noise**

- BatchNorm injects noise since the BN scale and shift depend on batch statistics (previous slide)

## Residual connections also smooth the loss surface

![The loss surface of a 56-layer CIFAR-10 ResNet **a)** with and **b)** without its skip connections (same network otherwise, both with BatchNorm). Figure 1 of Li et al. (2018).](figs/08-Residual-Networks-and-Normalization/loss-surface-residual-vs-none.png)

Residual connections and normalization both make the loss easier to descend; this figure isolates the skip connections. Compare the shattered-gradient picture from the start of the lecture.

*[H. Li, Z. Xu, G. Taylor, C. Studer, and T. Goldstein, "Visualizing the Loss Landscape of Neural Nets," arXiv:1712.09913](https://arxiv.org/abs/1712.09913v3)*

## LayerNorm in CNNs

- LayerNorm computes mean and variance across ALL features of a single sample
- For CNN output $(C, H, W)$, LN normalizes over $C \cdot H \cdot W$ values
- Same $\mu$ and $\sigma$ used for all channels and spatial positions
- Learned $\gamma$, $\delta$ applied per feature dimension
- No batch statistics, so no train/inference mismatch and no batch-size dependence. But LN over $C \cdot H \cdot W$ performs worse than BatchNorm or GroupNorm on ResNet-50 ([Wu & He, 2018](https://arxiv.org/abs/1803.08494)); when batches must be small (detection, segmentation, video) the usual choice is **GroupNorm**, the middle ground: normalize over groups of channels.

# Common residual architectures

With residual connections and batch normalization in hand, we look at
architectures built from them.


## ResNet (2015)

**ResNet Block**

![](figs/08-Residual-Networks-and-Normalization/resnet-block.png)

**Bottleneck Residual**

![](figs/08-Residual-Networks-and-Normalization/resnet-bottleneck-block.png)

*[K. He, X. Zhang, S. Ren, and J. Sun, "Deep Residual Learning for Image Recognition," arXiv:1512.03385, Dec. 2015](http://arxiv.org/abs/1512.03385)*.
The ordering drawn (BN $\rightarrow$ ReLU $\rightarrow$ Conv, the *pre-activation* block) is from the follow-up
*[He et al., "Identity Mappings in Deep Residual Networks," arXiv:1603.05027, 2016](https://arxiv.org/abs/1603.05027)*;
the original block was Conv $\rightarrow$ BN $\rightarrow$ ReLU with a ReLU after the addition.


## ResNet-200 (2016) for ImageNet Classification

![](figs/08-Residual-Networks-and-Normalization/resnet200-imagenet.png)

*[K. He, X. Zhang, S. Ren, and J. Sun, "Identity Mappings in Deep Residual Networks," arXiv:1603.05027, 2016](https://arxiv.org/abs/1603.05027)*.
ResNet-200 and the pre-activation bottleneck block drawn in the inset are from this second paper, not the original 2015 ResNet paper.

## ImageNet History

![](figs/08-Residual-Networks-and-Normalization/imagenet-history.png)

## DenseNet

![Figure from UDL.](figs/08-Residual-Networks-and-Normalization/densenet-udl.png)


![Figure from paper. Figure 1: A 5-layer dense block with a growth rate of $k = 4$. Each layer takes all preceding feature-maps as input.](figs/08-Residual-Networks-and-Normalization/densenet-paper.png)


*[Huang, G., Liu, Z., Van Der Maaten, L., & Weinberger, K. Q. (2017). Densely connected convolutional networks. IEEE/CVF Computer Vision & Pattern Recognition, 4700–4708. arXiv:1608.06993](https://arxiv.org/abs/1608.06993)*

## U-Net (2015)

![](figs/08-Residual-Networks-and-Normalization/unet.png)

*[Ronneberger, O., Fischer, P., & Brox, T. (2015). U-Net: Convolutional networks for biomedical image segmentation. International Conference on Medical Image Computing and Computer-Assisted Intervention, 234–241. arXiv:1505.04597](https://arxiv.org/abs/1505.04597)*


## U-Net Results

![Segmentation using U-Net in 3D. a) Three slices through a 3D volume of mouse cortex taken by scanning electron microscope. b) A single U-Net is used to classify voxels as being inside or outside neurites. Connected regions are identified with different colors. c) For a better result, an ensemble of five U-Nets is trained, and a voxel is only classified as belonging to the cell if all five networks agree. Adapted from Falk et al. (2019). (Prince, *Understanding Deep Learning*, Fig. 11.11)](figs/08-Residual-Networks-and-Normalization/unet-3d-segmentation.png)

## Stacked hourglass networks for Pose Estimation

![](figs/08-Residual-Networks-and-Normalization/stacked-hourglass-pose.png)

*[Newell, A., Yang, K., & Deng, J. (2016). Stacked hourglass networks for human pose estimation. European Conference on Computer Vision, 483–499. arXiv:1603.06937](https://arxiv.org/abs/1603.06937)*

## Feature Pyramid Networks

![](figs/08-Residual-Networks-and-Normalization/fpn-variants.png)


Figure 1. (a) Using an image pyramid to build a feature pyramid. Features are
computed on each of the image scales independently, which is slow. (b) Recent
detection systems have opted to use only single scale features for faster
detection. (c) An alternative is to reuse the pyramidal feature hierarchy
computed by a ConvNet as if it were a featurized image pyramid. **(d) Our proposed
Feature Pyramid Network (FPN)** is fast like (b) and (c), but more accurate. In
this figure, feature maps are indicate by blue outlines and thicker outlines
denote semantically stronger features.


*[T.-Y. Lin, P. Dollar, R. Girshick, K. He, B. Hariharan, and S. Belongie, "Feature Pyramid Networks for Object Detection," in 2017 IEEE Conference on Computer Vision and Pattern Recognition (CVPR), Honolulu, HI: IEEE, Jul. 2017, pp. 936–944. doi: 10.1109/CVPR.2017.106; arXiv:1612.03144](https://arxiv.org/abs/1612.03144)*

## Feature Pyramid Networks (cont.)

![Figure 2. Top: a top-down architecture with skip connections, where predictions are made on the finest level (e.g., [28]). Bottom: our model that has a similar structure but leverages it as a *feature pyramid*, with predictions made independently at all levels.](figs/08-Residual-Networks-and-Normalization/fpn-top-down.png)


![Figure 3. A building block illustrating the lateral connection and the top-down pathway, merged by addition.](figs/08-Residual-Networks-and-Normalization/fpn-building-block.png)


*[T.-Y. Lin, P. Dollar, R. Girshick, K. He, B. Hariharan, and S. Belongie, "Feature Pyramid Networks for Object Detection," in 2017 IEEE Conference on Computer Vision and Pattern Recognition (CVPR), Honolulu, HI: IEEE, Jul. 2017, pp. 936–944. doi: 10.1109/CVPR.2017.106; arXiv:1612.03144](https://arxiv.org/abs/1612.03144)*

# Normalization in transformers

Every architecture above is a residual network with BatchNorm. The transformer
you will meet in two lectures is also a residual network (the same unravelled,
variance-growing sum), but it normalizes along a different axis, usually drops
the mean, and has moved the normalization to a different place in the block.
BatchNorm is now the exception, not the rule, so this section covers what a
2026 language model actually does. The notation is the BatchNorm notation
above ($m, s, \gamma, \delta$); papers and PyTorch write the learned scale and
shift as $\gamma, \beta$ (`weight`, `bias`).


## LayerNorm: normalize each token over its features

A transformer carries $N$ tokens, each a vector $\mathbf{x}_n \in \mathbb{R}^D$.
LayerNorm computes the statistics of *one token* over its $D$ features:

$$
\begin{aligned}
m_n &= \frac{1}{D} \sum_{d=1}^{D} x_{nd} \\
s_n &= \sqrt{\frac{1}{D} \sum_{d=1}^{D} (x_{nd} - m_n)^2} \\
x_{nd} &\leftarrow \gamma_d \, \frac{x_{nd} - m_n}{s_n + \epsilon} + \delta_d
\end{aligned}
$$

$\gamma_d, \delta_d$: one learned scale and shift per feature ($2D$ parameters per LayerNorm), shared by all tokens. Papers write them $\gamma, \beta$.


**Why not BatchNorm?**

- The statistics never touch other tokens or other sequences: nothing to store, same computation at training and inference
- Sequences have different lengths and padding, which would corrupt batch statistics
- At generation time the "batch" is one new token
- Per-device batches are small in distributed training

Same recipe as BatchNorm, different axis: *per feature over the batch* became *per token over the features*.


*[J. L. Ba, J. R. Kiros, and G. E. Hinton, "Layer Normalization," arXiv:1607.06450, 2016](https://arxiv.org/abs/1607.06450)*

## RMSNorm: drop the mean

[Zhang & Sennrich (2019)](https://arxiv.org/abs/1910.07467) asked which part of
LayerNorm matters. Hypothesis: the *re-scaling* invariance, not re-centering.
So keep only the root mean square:

$$
\begin{aligned}
r_n &= \sqrt{\frac{1}{D} \sum_{d=1}^{D} x_{nd}^2} \\
x_{nd} &\leftarrow \gamma_d \, \frac{x_{nd}}{r_n + \epsilon}
\end{aligned}
$$

- No $m_n$, no $\delta_d$: one reduction instead of two, $D$ parameters instead of $2D$
- Same quality as LayerNorm in their experiments, with 7–64% less running time depending on the model


```python
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):   # x: (..., d)
        r = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(r + self.eps)
        return x * self.gamma
```

Implementations put $\epsilon$ inside the root, as here.


**Why it replaced LayerNorm:** cheaper for the same quality, and normalization is memory-bound (a full read and write of the activations), so the saving is real at scale. T5 (2020) adopted it, [Llama (2023)](https://arxiv.org/abs/2302.13971) made it the default, and Qwen, Gemma, Mistral and DeepSeek all use it.

*[B. Zhang and R. Sennrich, "Root Mean Square Layer Normalization," NeurIPS 2019, arXiv:1910.07467](https://arxiv.org/abs/1910.07467)*


## Pre-norm vs. post-norm: where the norm sits

The original transformer normalized *after* the residual addition (**post-norm**);
most decoder-only models since GPT-2 normalize the *input of the branch* (**pre-norm**);
Gemma 2/3 and OLMo 2 (next slides) are the exceptions:

**Post-norm** (Vaswani et al. 2017, BERT)
$$
\mathbf{h}_{l+1} = \mathrm{Norm}\big[\mathbf{h}_l + \mathbf{f}_l[\mathbf{h}_l]\big]
$$

- The residual stream is re-normalized every block: no variance growth
- But at initialization the gradient of the layers nearest the output is large and does not shrink with depth
- $\rightarrow$ needs a learning-rate **warmup** and a small learning rate, or training diverges


**Pre-norm** (GPT-2, Llama, Qwen, DeepSeek)
$$
\mathbf{h}_{l+1} = \mathbf{h}_l + \mathbf{f}_l\big[\mathrm{Norm}[\mathbf{h}_l]\big]
$$

- The residual stream is an un-normalized **sum**: $\mathbf{h}_L = \mathbf{h}_0 + \sum_{l} \mathbf{f}_l[\mathrm{Norm}[\mathbf{h}_l]]$, exactly the unravelled view from the start of the lecture
- Gradient at initialization scales as $1/\sqrt{L}$ for the last layers: well-behaved
- Xiong et al. trained pre-norm transformers *without warmup* at a larger learning rate, reaching the same loss in fewer steps


*[R. Xiong et al., "On Layer Normalization in the Transformer Architecture," ICML 2020, arXiv:2002.04745](https://arxiv.org/abs/2002.04745)*


## What pre-norm costs, and the 2025–26 placements

![One sub-block of a transformer layer under the four normalization placements in use. Each layer has two such sub-blocks, one for attention and one for the MLP.](figs/08-Residual-Networks-and-Normalization/norm-placement.png)

- Pre-norm's residual stream grows with depth, like the BatchNorm residual network earlier: block $l$ adds roughly one unit of variance to a stream of variance $l$, so late blocks make a smaller *relative* change
- **Sandwich** ([Gemma 2](https://arxiv.org/abs/2408.00118), [Gemma 3](https://arxiv.org/abs/2503.19786)): RMSNorm on the branch input *and* output
- **Reordered post-norm** ([OLMo 2](https://arxiv.org/abs/2501.00656)): normalize the branch output, but inside the residual so the stream itself is never normalized; adopted, with QK-norm, for training stability

## QK-norm: normalizing inside attention

The attention logit for one head is $\mathbf{q}_i^\top \mathbf{k}_j / \sqrt{d_k}$. During long training runs the query and key weights can grow until a few logits dominate: the softmax saturates to a one-hot, its gradient vanishes, and the loss spikes (*attention logit growth*).

**QK-norm** applies a normalization to each query and key vector before the dot product, over the head dimension $d_k$:

$$
\mathrm{softmax}\!\left[\frac{\mathrm{Norm}[\mathbf{q}_i]^\top \mathrm{Norm}[\mathbf{k}_j]}{\sqrt{d_k}}\right]
$$

With RMSNorm, $\|\mathrm{Norm}[\mathbf{q}]\| \le \sqrt{d_k}\,\max_d|\gamma_d|$, so the logit is bounded by the learned scales and cannot run away.


**Who uses it (2025–26)**

- [Gemma 3](https://arxiv.org/abs/2503.19786): QK-norm replaces Gemma 2's logit soft-capping
- [OLMo 2](https://arxiv.org/abs/2501.00656): QK-norm plus reordered post-norm, "to improve training stability"
- [Qwen3](https://arxiv.org/abs/2505.09388): removes the QKV bias of Qwen2 and adds QK-norm "to ensure stable training"

**Origin:** [Henry et al. 2020](https://arxiv.org/abs/2010.04245) (machine translation);
[Dehghani et al. 2023](https://arxiv.org/abs/2302.05442) used it to train a 22B-parameter ViT after logit divergence;
[Wortsman et al. 2023](https://arxiv.org/abs/2309.14322) identified attention logit growth as a scale-dependent instability that QK-norm removes.


## BatchNorm, LayerNorm, RMSNorm side by side

![](figs/08-Residual-Networks-and-Normalization/norm-axes.png)

| | BatchNorm | LayerNorm | RMSNorm |
|---|---|---|---|
| Statistics over | the batch (and space), per feature | the $D$ features, per token | the $D$ features, per token |
| Computed | $m_h, s_h$ | $m_n, s_n$ | $r_n$ (no mean) |
| Learned | $\gamma_h, \delta_h$ | $\gamma_d, \delta_d$ | $\gamma_d$ |
| Train vs. inference | different (running averages) | same | same |
| Where | ResNet, CNNs | original transformer, BERT, GPT-2, ViT | Llama, Qwen, Gemma, DeepSeek |


The three normalizations on one toy activation tensor of shape
(batch, tokens, features), with $\gamma = 1, \delta = 0$ (their initial
values). Each output is checked along the axis its statistics were computed
over: BatchNorm and LayerNorm give mean 0 and standard deviation 1 there,
RMSNorm gives RMS 1 but leaves the mean alone. The last lines change every
*other* sequence in the batch and show that only BatchNorm's output for
sequence 0 moves.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
B, N, D = 4, 5, 8                 # sequences, tokens per sequence, features
eps = 1e-6
x = rng.normal(2.0, 3.0, size=(B, N, D))

gamma = np.ones(D)                # learned scale per feature (initial value)
delta = np.zeros(D)               # learned shift per feature (initial value)

def batch_norm(x):                # per feature, over the batch and the tokens
    m = x.mean(axis=(0, 1), keepdims=True)
    s = x.std(axis=(0, 1), keepdims=True)
    return gamma * (x - m) / (s + eps) + delta

def layer_norm(x):                # per token, over its D features
    m = x.mean(axis=-1, keepdims=True)
    s = x.std(axis=-1, keepdims=True)
    return gamma * (x - m) / (s + eps) + delta

def rms_norm(x):                  # per token, over its D features, no mean
    r = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True))
    return gamma * x / (r + eps)

norms = [("BatchNorm", batch_norm, (0, 1), "batch and tokens", 2 * D),
         ("LayerNorm", layer_norm, -1, "features", 2 * D),
         ("RMSNorm", rms_norm, -1, "features", D)]
for name, f, axis, over, n_params in norms:
    y = f(x)
    mean, std = y.mean(axis=axis), y.std(axis=axis)
    rms = np.sqrt(np.mean(y ** 2, axis=axis))
    print(f"{name:9} over the {over:16}: max |mean| = {np.abs(mean).max():.2f}, "
          f"std in [{std.min():.2f}, {std.max():.2f}], RMS in [{rms.min():.2f}, {rms.max():.2f}]; "
          f"learned parameters: {n_params}")

# Replace every other sequence in the batch and see whose output for sequence 0 moves
x2 = x.copy()
x2[1:] = rng.normal(-5.0, 1.0, size=(B - 1, N, D))
print()
for name, f, *_ in norms:
    change = np.abs(f(x2)[0] - f(x)[0]).max()
    print(f"{name:9}: output for sequence 0 changes by up to {change:.3f}")

# Summary

What to take away from this lecture; the recap deck and the knowledge check
are built from this list.


## Summary

- Beyond about 20 layers a plain sequential network gets *worse*, even on training error: its gradient $\partial y / \partial x$ becomes uncorrelated noise (shattered gradients), so a gradient step lands somewhere with a completely different gradient.
- A residual block computes $\mathbf{h}_{k+1} = \mathbf{h}_k + \mathbf{f}_k[\mathbf{h}_k]$. Unravelled, a network of $K$ blocks is a sum over $2^K$ paths, an ensemble of networks of every depth; short paths keep the gradient well-behaved.
- Order of operations: end the branch with a linear layer, and start it with one if the resolution must change; a branch that starts with ReLU on the residual stream throws away its negative half.
- Adding the branch back in doubles the variance at every block, so it grows exponentially with depth unless something renormalizes.
- BatchNorm: per hidden unit, $m_h, s_h$ over the batch, $\hat{h} = (h - m_h)/(s_h + \epsilon)$, then $\gamma \hat{h} + \delta$ with learned $\gamma, \delta$. With $\gamma = 1, \delta = 0$ the variance grows linearly ($k$ after $k$ blocks) rather than exponentially.
- At inference BatchNorm uses running averages of $m_h, s_h$ and folds into the preceding convolution; the batch-statistic noise at training time is a regularizer. Call `model.eval()`.
- BatchNorm also lets you use a higher learning rate because it smooths the loss surface (Santurkar et al.), not because of "internal covariate shift".
- ResNet (bottleneck blocks, pre-activation ordering from He et al. 2016), DenseNet (concatenate instead of add), U-Net and hourglass (skip connections across an encoder-decoder), FPN (top-down pathway with lateral connections) are all residual designs.
- LayerNorm normalizes one token over its $D$ features ($m_n, s_n$, learned $\gamma_d, \delta_d$); it needs no batch, so it behaves identically at training and inference, which is why transformers use it instead of BatchNorm.
- RMSNorm keeps only the scale: $x_{nd} \leftarrow \gamma_d \, x_{nd} / (r_n + \epsilon)$ with $r_n$ the root mean square over features. Same quality, cheaper; the default since Llama.
- Pre-norm ($\mathbf{h} + \mathbf{f}[\mathrm{Norm}[\mathbf{h}]]$) gives well-scaled gradients at initialization and trains without warmup; post-norm ($\mathrm{Norm}[\mathbf{h} + \mathbf{f}[\mathbf{h}]]$) needs warmup. Pre-norm's residual stream grows with depth, so Gemma 2/3 add a post-norm too and OLMo 2 normalizes the branch output inside the residual.
- QK-norm normalizes each query and key over the head dimension before the dot product, bounding the attention logits; Gemma 3, OLMo 2 and Qwen3 use it to prevent loss spikes.